In [ ]:
import os
import requests

# ---- Konfiguration ----
GITHUB_RAW_BASE = "https://github.com/Fabric-Kompetenzteam/Fabric_Blueprint"
FILES = {
    # tabellenname : relativer pfad im repo
    "customers": "/blob/main/Data/datafile.parquet"
}

# Zielordner im Lakehouse Files
LAKEHOUSE_FILES_BASE = "file:/lakehouse/default/Files"

# Optional: Token für private Repos (als Notebook-Parameter oder Secret)
  # z.B. aus Secret Store ziehen, wenn vorhanden

headers = {}
if GITHUB_TOKEN:
    headers["Authorization"] = f"token {GITHUB_TOKEN}"

def download_to_lakehouse(raw_url: str, lakehouse_file_path: str):
    # lakehouse_file_path z.B. file:/lakehouse/default/Files/raw/customers/customers.parquet
    local_path = lakehouse_file_path.replace("file:", "")  # -> /lakehouse/default/Files/...
    os.makedirs(os.path.dirname(local_path), exist_ok=True)

    with requests.get(raw_url, headers=headers, stream=True, timeout=120) as r:
        r.raise_for_status()
        with open(local_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)

for table, rel_path in FILES.items():
    raw_url = f"{GITHUB_RAW_BASE}/{rel_path}"
    target = f"{LAKEHOUSE_FILES_BASE}/{table}/{os.path.basename(rel_path)}"
    print(f"Downloading {raw_url} -> {target}")
    download_to_lakehouse(raw_url, target)

print("Done.")


StatementMeta(, 090d5997-a028-40c0-8349-45af80ecc4c4, 3, Finished, Available, Finished)

Done.


In [1]:
import os
import requests
from urllib.parse import quote

# ========= Konfiguration =========
OWNER = "Fabric-Kompetenzteam"
REPO  = "Fabric_Blueprint"
BRANCH = "main"
GITHUB_ROOT_FOLDER = "Data"   # Ordner im Repo, den du kopieren willst

# Ziel im Lakehouse Files (ohne file:)
LAKEHOUSE_TARGET_ROOT = "/lakehouse/default/Files/Data"

session = requests.Session()
session.headers.update({"Accept": "application/vnd.github+json"})


def gh_contents(path: str):
    url = f"https://api.github.com/repos/{OWNER}/{REPO}/contents/{quote(path)}"
    r = session.get(url, params={"ref": BRANCH}, timeout=120)
    r.raise_for_status()
    return r.json()


def download_file(download_url: str, local_path: str):
    os.makedirs(os.path.dirname(local_path), exist_ok=True)
    with session.get(download_url, stream=True, timeout=300) as r:
        r.raise_for_status()
        with open(local_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)


def sync_folder(repo_path: str, lakehouse_path: str):
    items = gh_contents(repo_path)
    if isinstance(items, dict) and items.get("type") == "file":
        items = [items]

    for it in items:
        t = it.get("type")
        name = it.get("name")

        if t == "dir":
            sync_folder(it["path"], os.path.join(lakehouse_path, name))

        elif t == "file":
            dl = it.get("download_url")
            if not dl:
                print(f"Skip (no download_url): {it['path']}")
                continue

            target = os.path.join(lakehouse_path, name)
            print(f"Downloading {it['path']} -> {target}")
            download_file(dl, target)

        else:
            print(f"Skip unknown type {t}: {it.get('path')}")

# ========= Start =========
sync_folder(GITHUB_ROOT_FOLDER, LAKEHOUSE_TARGET_ROOT)
print("Done.")


StatementMeta(, 9282ad86-d005-48aa-9ff5-33d607e8cc13, 3, Finished, Available, Finished)

Done.


In [2]:
base = "file:/lakehouse/default/Files/Data"

tables = [
    "employees", "passengers", "shifts", "taxizones",
    "vehicle_categories", "vehicle_models", "vehicles", "yellowtaxitrips"
]

spark.sql("CREATE SCHEMA IF NOT EXISTS raw")

for t in tables:
    df = spark.read.parquet(f"{base}/{t}")
    df.write.format("delta").mode("overwrite").saveAsTable(f"raw.{t}")
    print("Imported:", t)


StatementMeta(, 9282ad86-d005-48aa-9ff5-33d607e8cc13, 4, Finished, Available, Finished)

Imported: employees
Imported: passengers
Imported: shifts
Imported: taxizones
Imported: vehicle_categories
Imported: vehicle_models
Imported: vehicles
Imported: yellowtaxitrips


In [3]:
%%sql
SELECT COUNT(*) from raw.yellowtaxitrips

StatementMeta(, 9282ad86-d005-48aa-9ff5-33d607e8cc13, 5, Finished, Available, Finished)

<Spark SQL result set with 1 rows and 1 fields>